# CS-228 YOLO Distillation Training (Colab H100/A100 Optimized)
**Instructions:**
1. Go to `Runtime > Change runtime type` and select **A100** or **L4** (or T4 if standard).
2. Ensure your zipped `custom_datasets_only.zip` and the python code (`CS-228-Project/`) are uploaded to Google Drive.
3. Run the cells below sequentially.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install Dependencies
!pip install ultralytics opencv-python numpy pyyaml

In [ ]:
# 3. Copy Code from Drive to fast local storage
# Update this path to wherever your project code lives in Drive:
PROJECT_DRIVE_PATH = "/content/drive/MyDrive/CS-228-Project"

!cp -r "$PROJECT_DRIVE_PATH" /content/CS-228-Project
%cd /content/CS-228-Project

# 4. Copy Datasets to fast local NVMe storage and extract
# Update this path to where you uploaded custom_datasets_only.zip
DATASET_ZIP_PATH = "/content/drive/MyDrive/CS-228-Project/custom_datasets_only.zip"

!mkdir -p datasets/
!cp "$DATASET_ZIP_PATH" /content/
!unzip -q /content/custom_datasets_only.zip -d datasets/

# Clean up zip to save disk space
!rm /content/custom_datasets_only.zip

In [ ]:
# 5. Force Download COCO directly onto the fast Colab network (Takes ~2 mins here)
import os
from ultralytics.data.utils import check_det_dataset
from ultralytics import settings

# Tell Ultralytics to use our local datasets folder instead of global
settings.update({'datasets_dir': '/content/CS-228-Project/datasets'})

print("Downloading COCO locally...")
check_det_dataset('coco.yaml')
print("Finished COCO Download!")

## Training the Models
We can now drastically increase the batch size because Colab GPUs have 40GB - 80GB of VRAM (unlike laptops with 8GB).
A batch size of `128` or `256` will train exponentially faster.

In [ ]:
# 6. Train the Teachers!
!python training/train_teachers.py --epochs 150 --batch 128 --device 0 --model yolov26 --fraction 0.1 --cache


In [ ]:
# 7. Run Distillation Batch Training
!python batch_train.py --model yolov26 --cache


In [ ]:
# 7.5 Compare YOLOv26 Teacher and Student on Testset
!pip install plotly pandas
!python compare_models.py --models models/teachers/yolov26_teacher.pt project/student_yolo26n_cans_best.pt


In [ ]:
# 8. Save EVERYTHING permanently back to your Google Drive
# Because Colab deletes /content/ when it disconnects, we have to copy the weights out.
!cp -r /content/CS-228-Project/models "$PROJECT_DRIVE_PATH/"
!cp -r /content/CS-228-Project/project "$PROJECT_DRIVE_PATH/"
!cp -r /content/CS-228-Project/runs "$PROJECT_DRIVE_PATH/"

print("✅ All trained models and logs successfully saved to Google Drive!")